# `commodities(base, start, end=None, quote="USD")`

Daily prices for 28 physical commodities, from FMP via `commodities.prices`, in USD or any currency `forex()` can price. Every cell runs against prod, read-only, as `readonly_user`. Change the arguments and re-run any cell.

In [ ]:
from datetime import date, timedelta

import polars as pl
from IPython.display import Markdown

from arora_data.commodities import ALIASES, COMMODITIES, CommodityBar
from arora_data.dates import today_et
from arora_data.frames import to_df
from arora_data.sync import commodities


def show(rows):
    """One result or a list of results as a Markdown table."""
    rows = [r.model_dump() for r in (rows if isinstance(rows, list) else [rows])]
    if not rows:
        return Markdown("0 rows")
    head = list(rows[0])
    lines = ["|".join(head), "|".join("---" for _ in head)]
    lines += ["|".join(str(r[c]).replace("|", "/") for c in head) for r in rows]
    return Markdown(f"{len(rows)} rows\n\n" + "\n".join(lines))

## The 28 commodities
Call them by root code, name or alias. Each price is for one of `unit`, from the CME, ICE and LME contract specs. The table's financial futures (ES, NQ, YM, RTY, ZB, ZN, ZF, ZT, ZQ, DX) and the micro gold and silver contracts (MGC, SIL, which mirror GC and SI) are not served.

In [ ]:
pl.DataFrame([{"code": c, "name": n, "symbol": s, "unit": u} for c, (s, u, n) in COMMODITIES.items()])

## By code, name or alias
A name or alias works anywhere a code does, in any case and spacing. Every bar carries the code and the primary name. Words that could mean two commodities (`oil`, `gas`, `cattle`) are not aliases.

In [ ]:
week = today_et() - timedelta(days=7)
assert commodities("GC", week) == commodities("gold", week) == commodities("  Gold ", week)
show(commodities("crude", week)[-2:] + commodities("Wheat", week)[-2:])

In [ ]:
pl.DataFrame({"alias": list(ALIASES), "code": list(ALIASES.values())})

## In USD, the default
FMP daily bars as stored (`route` = stored): open, high, low, close and volume. `quote_date` is blank because no rate is used.

In [ ]:
show(commodities("GC", today_et() - timedelta(days=7)))

## A symbol FMP quotes in cents
Corn is stored as ZCUSX, in US cents per bushel. It comes back in dollars.

In [ ]:
show(commodities("ZC", today_et() - timedelta(days=7)))

## In another currency
The USD price times `forex("USD", "EUR")` that day (`route` = crossed). Open uses the day's opening rate, close its closing rate. High and low are blank: the commodity and the rate peak at different moments. Volume is contracts of the commodity, so it stays.

In [ ]:
show(commodities("CL", today_et() - timedelta(days=7), quote="EUR"))

## Either way round a pair is stored
EUR is stored as EURUSD and JPY as USDJPY; `forex()` handles both, so any currency it prices works.

In [ ]:
start = today_et() - timedelta(days=7)
rows = [r for q in ["USD", "EUR", "JPY", "GBP", "INR", "CHF"] for r in commodities("GC", start, quote=q)]
to_df(rows, CommodityBar).pivot(on="unit", index="date", values="close")

## Days with no rate carry the last one
A commodity can trade on a day its currency pair has no bar. That day uses the pair's last close, for open and close alike, and `quote_date` names the day the rate came from. Here: every carried day for gold in EUR over the last five years.

In [ ]:
gc_eur = to_df(commodities("GC", today_et() - timedelta(days=5 * 365), quote="EUR"), CommodityBar)
carried = gc_eur.filter(pl.col("quote_date") != pl.col("date"))
print(f"{carried.height} of {gc_eur.height} days carried a rate")
carried.select("date", "quote_date", "open", "close").tail(10)

## The first days look back before start
Starting on a carried day still gives that day a price: the rate is looked up as far as 14 days before `start`.

In [ ]:
first = carried["date"][-1]
show(commodities("GC", first, first + timedelta(days=3), quote="EUR"))

## Before the currency's history, days are skipped
FMP's forex history starts on 1999-01-01, so gold in EUR starts there while gold in USD goes back further. Nothing is invented for the earlier days.

In [ ]:
span = date(1998, 12, 24), date(1999, 1, 8)
print("USD:", [r.date.isoformat() for r in commodities("GC", *span)])
print("EUR:", [r.date.isoformat() for r in commodities("GC", *span, quote="EUR")])

## `commodities("ALL", start, end, quote)`
All 28 in one currency, ordered by code then date. Each symbol is read once, and the currency rate once.

In [ ]:
to_df(commodities("ALL", today_et() - timedelta(days=7), quote="JPY"), CommodityBar).pivot(
    on="commodity", index="date", values="close")

## Full history in one call
Gold in USD from the start of FMP's history. There is no row limit.

In [ ]:
gold = to_df(commodities("GC", date(1970, 1, 1)), CommodityBar)
print(f"{gold.height:,} days, {gold['date'].min()} to {gold['date'].max()}")
(
    gold.group_by(pl.col("date").dt.year().alias("year"), maintain_order=True)
    .agg(pl.col("date").last().alias("last_day"), pl.col("close").last())
    .tail(10)
)

## Bad input
An unserved code (a financial future, a micro contract, an FMP symbol instead of a root code), a name or alias that isn't on the list (`oil` could be WTI or Brent), `quote="ALL"`, a currency with no USD pair, or a start after the end each raise a clear error.

In [ ]:
week = today_et() - timedelta(days=7)
for base, quote in [("ES", "USD"), ("MGC", "USD"), ("GCUSD", "USD"), ("oil", "USD"), ("GC", "ALL"), ("GC", "XAU")]:
    try:
        commodities(base, week, quote=quote)
    except ValueError as err:
        print(f"{base}/{quote}: {err}")
try:
    commodities("GC", today_et(), week)
except ValueError as err:
    print(f"start after end: {err}")